# Cati 토크나이저 — Kaggle

**학습은 Colab에서 합니다** (`cati_colab.ipynb`). Kaggle에는 TPU 옵션이 없어서요.
이 노트북은 토크나이저만 만듭니다 — 귀한 Colab 시간을 30분 아끼려고요.

건너뛰어도 됩니다. Colab 노트북이 토크나이저가 없으면 알아서 만듭니다.

### 하는 법
1. Settings → Accelerator **None** · Internet **On**
2. **Save Version → Save & Run All** → 창 닫기
3. 30분 뒤 Output의 `tokenizer.json` 을 내려받아
   Google Drive의 `MyDrive/cati/` 에 넣기

In [ ]:
TOKENIZER_DOCS = 400_000

In [ ]:
# ══ 토크나이저 학습 ══
import os, shutil, socket, subprocess, sys
from pathlib import Path


def stream(cmd, check=False):
    """자식 프로세스 출력을 셀에 실시간으로 흘린다 (subprocess.run 은 안 보인다)."""
    p = subprocess.Popen([sys.executable, "-u", *cmd], stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    rc = p.wait()
    if check and rc != 0:
        raise SystemExit(f"실패 (종료 코드 {rc})")
    return rc

CATI = Path("/kaggle/working/Cati")
if CATI.exists():
    subprocess.run(["git", "-C", str(CATI), "pull", "-q"], check=False)
else:
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/foeplob11-code/Cati.git", str(CATI)], check=True)
os.chdir(CATI)
sys.path.insert(0, str(CATI))
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "tokenizers>=0.22", "datasets>=3.0"], check=False)

try:
    socket.create_connection(("huggingface.co", 443), timeout=10).close()
except OSError:
    raise SystemExit("인터넷이 꺼져 있다 → Settings → Internet → On")

TOK = Path("artifacts/tokenizer/tokenizer.json")
TOK.parent.mkdir(parents=True, exist_ok=True)
if not TOK.exists():
    stream(["scripts/train_tokenizer.py", "train", "--docs", str(TOKENIZER_DOCS)],
           check=True)

from tokenizers import Tokenizer
_t = Tokenizer.from_file(str(TOK))
_p = "고양이는 창가에 앉아 오래 밖을 바라보았다."
_r = len(_p) / len(_t.encode(_p).ids)
print(f"\nvocab {_t.get_vocab_size():,} · 한국어 {_r:.2f} 글자/토큰 "
      f"({'통과' if _r >= 2.0 else '미달 — 알려주세요'})")
print("기준선: SmolLM2(같은 vocab) 0.47 · Qwen3(vocab 3배) 1.39 · 목표 2.0 이상")

out = Path("/kaggle/working/tokenizer.json")
shutil.copy(TOK, out)
print(f"\n저장: {out}  ({out.stat().st_size/1e6:.2f} MB)")
print("\n다음: 이 파일을 내려받아 Google Drive의 MyDrive/cati/ 에 넣으세요.")
print("      그러면 Colab 노트북이 바로 씁니다.")